# Student Performance Analysis – Mathematics & Statistics

This Google Colab notebook analyses the repository's `student_dataset.csv`. It covers descriptive statistics, grouped summaries, correlation, simple linear regression, and visualizations.

## 1. Import libraries and load the dataset

The code first looks for the CSV in the current Colab session. If it is not present, it downloads the raw file from this GitHub repository.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

DATA_PATH = 'student_dataset.csv'
GITHUB_CSV_URL = 'https://raw.githubusercontent.com/SusmithaBR/raga-susmitha/main/student_dataset.csv'

if not os.path.exists(DATA_PATH):
    import urllib.request
    urllib.request.urlretrieve(GITHUB_CSV_URL, DATA_PATH)

df = pd.read_csv(DATA_PATH)
print('Dataset loaded successfully')
print('Rows:', df.shape[0], '| Columns:', df.shape[1])
display(df.head())


## 2. Understand and validate the data

In [ ]:
print('Dataset information:')
df.info()

print('\nMissing values in each column:')
display(df.isnull().sum().to_frame('Missing values'))

print('\nDuplicate rows:', df.duplicated().sum())
print('\nNumeric summary:')
display(df.describe())


## 3. Descriptive statistics

Calculate mean, median, mode, range, variance, standard deviation, minimum, and maximum for marks and study hours.

In [ ]:
def descriptive_summary(series):
    return pd.Series({
        'Count': series.count(),
        'Mean': series.mean(),
        'Median': series.median(),
        'Mode': ', '.join(map(str, series.mode().tolist())),
        'Minimum': series.min(),
        'Maximum': series.max(),
        'Range': series.max() - series.min(),
        'Variance': series.var(),
        'Standard deviation': series.std()
    })

stats = pd.DataFrame({
    'Marks': descriptive_summary(df['Marks']),
    'Hours_Study': descriptive_summary(df['Hours_Study'])
})
display(stats.round(2))


## 4. Department-wise mathematical summary

In [ ]:
department_summary = df.groupby('Department').agg(
    Student_Count=('Name', 'count'),
    Average_Marks=('Marks', 'mean'),
    Median_Marks=('Marks', 'median'),
    Highest_Marks=('Marks', 'max'),
    Lowest_Marks=('Marks', 'min'),
    Average_Study_Hours=('Hours_Study', 'mean')
).sort_values('Average_Marks', ascending=False)
display(department_summary.round(2))

print('Department with highest average marks:', department_summary['Average_Marks'].idxmax())


## 5. Gender-wise summary

In [ ]:
gender_summary = df.groupby('Gender').agg(
    Student_Count=('Name', 'count'),
    Average_Marks=('Marks', 'mean'),
    Average_Study_Hours=('Hours_Study', 'mean')
)
display(gender_summary.round(2))


## 6. Frequency distribution and performance categories

Classify marks into simple performance bands.

In [ ]:
bins = [0, 60, 70, 80, 90, 100]
labels = ['Below 60', '60–69', '70–79', '80–89', '90–100']
df['Performance_Band'] = pd.cut(df['Marks'], bins=bins, labels=labels, include_lowest=True, right=False)
frequency = df['Performance_Band'].value_counts(sort=False).rename_axis('Performance band').to_frame('Number of students')
display(frequency)


## 7. Correlation between study hours and marks

In [ ]:
correlation = df['Hours_Study'].corr(df['Marks'])
print(f'Pearson correlation between study hours and marks: {correlation:.3f}')
if correlation > 0.7:
    interpretation = 'Strong positive linear relationship.'
elif correlation > 0.3:
    interpretation = 'Moderate positive linear relationship.'
elif correlation >= -0.3:
    interpretation = 'Weak or no linear relationship.'
elif correlation >= -0.7:
    interpretation = 'Moderate negative linear relationship.'
else:
    interpretation = 'Strong negative linear relationship.'
print(interpretation)


## 8. Simple linear regression

Fit a straight-line model to estimate marks from study hours. This is an educational analysis of this small dataset, not a reliable prediction tool for other students.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

X = df[['Hours_Study']]
y = df['Marks']
model = LinearRegression()
model.fit(X, y)
predicted = model.predict(X)

print(f'Intercept: {model.intercept_:.2f}')
print(f'Slope (marks per additional study hour): {model.coef_[0]:.2f}')
print(f'R-squared on the same dataset: {r2_score(y, predicted):.3f}')
print(f'RMSE on the same dataset: {np.sqrt(mean_squared_error(y, predicted)):.2f}')

sample_hours = pd.DataFrame({'Hours_Study': [4, 5, 6]})
sample_hours['Estimated_Marks'] = model.predict(sample_hours[['Hours_Study']]).round(2)
display(sample_hours)


## 9. Visualizations

The following plots are generated directly from the dataset.

In [ ]:
sns.set_theme()

plt.figure(figsize=(9, 5))
sns.barplot(data=df, x='Department', y='Marks', estimator='mean', errorbar=None)
plt.title('Average Marks by Department')
plt.xlabel('Department'); plt.ylabel('Average marks')
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(6, 6))
df['Gender'].value_counts().plot(kind='pie', autopct='%1.1f%%', startangle=90)
plt.title('Gender Distribution')
plt.ylabel('')
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='Hours_Study', y='Marks', hue='Gender', s=90)
sns.regplot(data=df, x='Hours_Study', y='Marks', scatter=False, color='black')
plt.title('Study Hours vs Marks')
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x='Marks', bins=8, kde=True)
plt.title('Distribution of Student Marks')
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x='Department', y='Marks')
plt.title('Marks Distribution by Department')
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
ranked = df.sort_values('Marks', ascending=False).reset_index(drop=True)
plt.plot(ranked.index + 1, ranked['Marks'], marker='o')
plt.title('Student Marks Ranking')
plt.xlabel('Rank'); plt.ylabel('Marks')
plt.xticks(range(1, len(ranked) + 1))
plt.tight_layout(); plt.show()


## 10. Key findings

The following outputs summarize the main observations calculated from the dataset.

In [ ]:
top_student = df.loc[df['Marks'].idxmax()]
lowest_student = df.loc[df['Marks'].idxmin()]
print(f"Number of students: {len(df)}")
print(f"Overall average marks: {df['Marks'].mean():.2f}")
print(f"Highest marks: {top_student['Marks']} ({top_student['Name']}, {top_student['Department']})")
print(f"Lowest marks: {lowest_student['Marks']} ({lowest_student['Name']}, {lowest_student['Department']})")
print(f"Average study hours: {df['Hours_Study'].mean():.2f}")
print(f"Study-hours/marks correlation: {correlation:.3f}")


## Conclusion

This notebook applies descriptive statistics, grouped analysis, correlation, linear regression, and data visualization to the student performance dataset. Interpret all findings in the context of this small sample of 25 records.